## Load data from hf

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from datasets import load_dataset

In [ ]:
os.getenv("HF_TOKEN")

In [ ]:
df = load_dataset("viamr-project/reasoning-frame-hint", token=os.getenv("HF_TOKEN"))
df

In [ ]:
sample = df["train"][0]

# Hint

In [ ]:
from services.amr_hint.amr_hint import AMRHint

amr = """
(w / want-01
    :ARG0 (b / boy)
    :ARG1 (g / go-01
              :ARG0 b))
""".strip()

hint_gen = AMRHint()
hints = hint_gen.get_hints_json(amr)
print(hints)

In [ ]:
from services.amr_hint.prompt_builder import build_prompt

print(build_prompt(sample["sentence"], sample["amr"]))

In [ ]:
partial = hint_gen.get_hints_partial(amr, percentage=0.3)
print(partial)

In [ ]:
import json

# JSON (string) - phù hợp để đưa thẳng vào model
print(hint_gen.get_hints_json(amr, indent=2))

# Hoặc lấy dict Python (structured object)
structured = hint_gen.get_hints_structured(amr)
structured

# vLLM

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
import os
from vllm import SamplingParams

vllm_config = {
    "model": "Qwen/Qwen3.6-35B-A3B-FP8",
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.85,
    "max_model_len": 4096,
    
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repetition_penalty": 1.0,
    "max_tokens": 1024,
}

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sampling_params = SamplingParams(
    temperature=vllm_config["temperature"],
    top_p=vllm_config["top_p"],
    top_k=vllm_config["top_k"],
    min_p=vllm_config["min_p"],
    presence_penalty=vllm_config["presence_penalty"],
    repetition_penalty=vllm_config["repetition_penalty"],
    max_tokens=vllm_config["max_tokens"],
)

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")


try:
    vllm_model = LLM(
        model=vllm_config["model"],
        tensor_parallel_size=vllm_config["tensor_parallel_size"],
        gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
        max_model_len=vllm_config["max_model_len"],
        hf_token=os.getenv("HF_TOKEN")
    )
    print(f"✓ vLLM model loaded successfully: {vllm_config["model"]}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from vllm import LLM

prompt = "AMR trong NLP là gì?"

outputs = vllm_model.generate(prompt, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
print(generated_text)

In [ ]:
sample = {"amr":"(d / dream-01\n   :ARG0 (a / anyone\n            :ARG0-of (b / believe-01\n                        :ARG1 (t / that))))","sentence":"Anyone who believes that will happen is dreaming.","model_respose":["To parse the sentence \"Anyone who believes that will happen is dreaming\" into Abstract Meaning Representation (AMR), we break down the semantic structure as follows:\n\n1.  **Main Clause**: The core assertion is about a person (\"Anyone\") performing the action of \"dreaming\".\n    *   Entity: `x` is a `person`.\n    *   Relation: This person `w0` is the agent (`:agent`) of the action `dream-01`.\n    *   Tense: The main verb \"is dreaming\" is in the present continuous tense, implying current relevance. In AMR, we often capture the time or aspect via modifiers, but the core relation is `:arg0` (or `:agent`) linking the dreamer to the event.\n\n2.  **Relative Clause**: The phrase \"who believes that will happen\" modifies \"Anyone\" (the `person`).\n    *   Entity: The same `x` (the person) is also the believer.\n    *   Action: `believer-01` (often simplified to just `believe-01` in standard AMR framesets, though the hint uses `believe-01`). Let's stick to `believe-01`.\n    *   Believer (`:arg0`): The person `x`.\n    *   Content believed (`:arg1`): That \"it will happen\".\n\n3.  **Subordinate Clause (Content of Belief)**: The object of belief is the event \"that will happen\".\n    *   Event: `happen-01`.\n    *   Time/Tense: \"will happen\" indicates future tense. This can be represented with a `:time` or specific tense markers if available, but structurally it is an instance of `happen-01`.\n\n4.  **Connecting the Dots**:\n    *   We have a person `w0`.\n    *   `w0` is dreaming (`dream-01`). The dreamer argument for `dream-01` is `w0`.\n    *   `w0` believes something (`believe-01`). The believer argument is `w0`.\n    *   The thing believed is that something happens (`happen-01`).\n\nLet's refine the AMR graph structure:\n\n*   **Root**: Usually, the main predicate determines the root if no entity is explicitly topicalized as a named noun. Here, the sentence describes a state of being (\"is dreaming\"). Often, if the sentence focuses on the person, the person might be the root, or the event \"dream\" might be. However, looking at the provided hints, the root is suggested as `dream-01`. Let's verify if `dream-01` makes sense as the root.\n    *   If `dream-01` is the root: `(d / dream-01)`\n    *   Who is dreaming? `:arg0 (w / person)` where `w :name ...` no, `w` is just a person. Actually, `dream-01` usually takes `:arg0` as the dreamer and `:arg1` as the dream content. But here \"dreaming\" is used intransitively/metaphorically (\"is dreaming\" = \"has unrealistic hopes\"). In AMR, this meaning is often captured by `dream-01` with just the agent, or sometimes mapped to `wish-01` or similar, but given the hint `dream-01`, we follow that.\n    *   Wait, the hint says `root: dream-01`.\n    *   So, `(d / dream-01)`\n    *   `:arg0 (p / person)` -> The person is dreaming.\n    *   Now, describe the person `p`.\n    *   `p` has a property/state of believing.\n    *   `(b / believe-01)`\n    *   `:arg0 p` (The person believes).\n    *   `:arg1 (h / happen-01)` (That it happens).\n    *   Is there a connection between `dream` and `believe`?\n        *   Sentence: \"Anyone [who believes...] is dreaming.\"\n        *   This implies the person is *characterized* by dreaming. Or rather, the act of dreaming is predicated of the person defined by the relative clause.\n        *   Standard AMR parsing would link the person to both actions.\n        *   Structure:\n            ```\n            (d / dream-01\n              :arg0 (p / person\n                :mod (w / who) -- usually 'who' isn't a separate node, 'anyone' is just a determiner/quantifier, often implicit or represented by existence.\n                :example (b / believe-01\n                  :arg0 p\n                  :arg1 (h / happen-01))))\n            ```\n        *   However, \"Anyone who...\" is a universal quantification. AMR typically handles this by asserting the relationship holds for any such entity. We represent the entity and its attributes.\n        *   The \"dream\" here is likely metaphorical (having a delusion).\n        *   Let's look at the flow: Person -> Dreams. Person -> Believes. Believing -> Happen.\n\n    Let's construct the concrete nodes:\n    1.  `(d / dream-01)` - Root from hints.\n    2.  `:arg0 (p / person)` - The dreamer.\n    3.  The person `p` is restricted by \"who believes that will happen\".\n    4.  Add a `believe-01` instance connected to `p`.\n        *   `(b / believe-01 :arg0 p :arg1 (h / happen-01))`\n    5.  How do we attach this belief to the person or the dream?\n        *   Typically, modifiers on the subject like \"who...\" are attached to the subject entity.\n        *   So, `p` is linked to `b`.\n        *   Is there a direct link between `dream` and `believe`? Not necessarily. The sentence says \"The person who believes X is dreaming.\" This means: Exists a person P. P believes H. P dreams.\n        *   Since `dream-01` is the root, `P` is its arg0. And `P` also participates in `believe-01`.\n\n    Refined Graph:\n    ```penman\n    (d / dream-01\n       :arg0 (p / person\n          :mod (w / who) ? No, 'who' is a relative pronoun.\n          :instance-of ? No.\n          )\n    ```\n    Actually, simpler:\n    `(d / dream-01 :arg0 (p / person :belief (b / believe-01 :arg0 p :arg1 (h / happen-01))))` -> AMR doesn't use complex nesting tags like `:belief` usually. It links entities.\n\n    Correct linkage:\n    `(d / dream-01 :arg0 p)`\n    `(p / person)`\n    `(b / believe-01 :arg0 p :arg1 h)`\n    `(h / happen-01)`\n\n    We need to connect `b` to `p`.\n    \n    Final check on semantics:\n    \"Dreaming\" can sometimes map to `think-01` or `believe-01` if interpreted as holding a false belief, but the word is \"dreaming\". The hint confirms `dream-01`.\n    \n    Does \"dream-01\" take `:arg1`?\n    Hint: `\"used_args\": [\"ARG0\"]` for dream-01. So no content of the dream.\n    Hint: `\"used_args\": [\"ARG0\", \"ARG1\"]` for believe-01.\n    \n    So:\n    Node `d`: `dream-01`\n    Node `p`: `person`\n    Relation: `d.arg0 -> p`\n    \n    Node `b`: `believe-01`\n    Node `p`: `person`\n    Relation: `b.arg0 -> p`\n    \n    Node `h`: `happen-01`\n    Relation: `b.arg1 -> h`\n\n    There is no explicit temporal marker in the AMR standard base set other than generic `:mof` or relying on context, but \"will\" suggests future for happen. `happen-01` doesn't strongly require a tense node unless specified.\n\n    One detail: \"Anyone\". In AMR, indefinite/universal quantifiers are often ignored or handled by the variable existence. We just say there is a person.\n\n    Structure:\n    ```\n    (d / dream-01\n      :arg0 (p / person\n        :example (b / believe-01\n          :arg0 p\n          :arg1 (h / happen-01))))\n    ```\n    Wait, `:example` is not a standard edge label for modifying a subject with a relative clause in all AMR versions. Usually, we just list the instances and let the connections define them, or we don't nest.\n    \n    Standard way to write \"The person who believes X\" where Person is arg0 of main verb:\n    `(d / dream-01 :arg0 p)`\n    `(p / person)`\n    `(b / believe-01 :arg0 p :arg1 h)`\n    `(h / happen-01)`\n    \n    How to indicate `b` restricts `p`?\n    Often, if `p` is the topic, we might write:\n    `(p / person :rel (b / believe-01 ...))`? No, `:rel` is for adjectival relations or specific templates.\n    \n    A common pattern for \"NP who VP\" where NP is an arg of the main verb:\n    Main Verb (Root) --arg0--> Person\n    Person --[implicit/structural]--> Believe\n    \n    In linear Penman notation without nesting syntax (which is clearer):\n    \n    (d / dream-01\n      :arg0 p)\n    \n    (p / person)\n    \n    (b / believe-01\n      :arg0 p\n      :arg1 h)\n    \n    (h / happen-01)\n    \n    However, AMR graphs are connected. If I output separate components, it's technically one graph if `p` is referenced.\n    \n    Let's try to format it cleanly.\n\n<amr>\n(d / dream-01\n  :arg0 (p / person\n    :mod (b / believe-01\n      :arg0 p\n      :arg1 (h / happen-01)))\n</amr>"]}

sample["model_respose"][0]

In [ ]:
print(sample["model_respose"][0])